# DTU Social Data (2026) - Final Project Explainer
**Group Members:** Andrea De Pascale, [Groupmate 1 Name], [Groupmate 2 Name]

---

## 1. Motivation

**What is your dataset?**
Our analysis relies on three primary datasets:
1. **HFUDD16** (Statistics Denmark): Contains highly granular data on education levels and socioeconomic status (employed, unemployed, enrolled) across Danish municipalities.
2. **IFOR35** (Statistics Denmark): Contains historical data on the average income distributed by deciles and municipalities.
3. **Company Data**: A supplemental dataset tracking the locations and industries of the main companies operating in Denmark.

**Why did you choose these particular datasets?**
We chose these datasets because, when combined, they provide a holistic view of a municipality's economic health. Instead of just looking at raw income, linking HFUDD16 and IFOR35 allows us to see *who* is making that money (education level) and *what* the local workforce looks like. Adding the company data allows us to identify the corporate drivers behind municipal wealth. 

**What was your goal for the end user's experience?**
Our goal was to create an interactive "Martini Glass" experience. We wanted to first guide the user through our top-level findings (steady national growth, uniform demographics, and localized anomalies), and then provide them with an interactive dashboard where they could independently explore the socioeconomic footprint of any municipality they choose.

In [1]:
# Import necessary libraries for the analysis
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

# Suppress warnings for cleaner notebook presentation
import warnings
warnings.filterwarnings('ignore')

## 2. Basic stats. Let's understand the dataset better

### Data Cleaning and Preprocessing
To create a unified dataset for our interactive visualizations, we had to perform several preprocessing steps:
* **String Normalization:** The municipality names in the datasets had slight variations (e.g., "København" vs "København Kommune"). We stripped suffixes and standardized characters (æ, ø, å) to ensure accurate merging.
* **Merging Data:** We performed an inner join on the `HFUDD16` and `IFOR35` datasets using `Municipality` and `Year` as the primary keys.
* **Ratio Calculations:** For the ternary plot, raw population counts were less useful than compositional data. We aggregated the data by municipality to calculate the specific *ratios* of Employed, Unemployed, and Enrolled individuals.

*(Groupmates: Add your specific preprocessing steps here regarding the company data and education flow data).*

### Exploratory Data Analysis (EDA)
During our initial EDA, we tracked the baseline national average income from 2008 to 2024. We found a very steady, linear increase across almost all municipalities, which aligns perfectly with standard economic inflation and national wage adjustments.

In [2]:
# Load your merged dataset
df = pd.read_csv('src/Part1/data/Part1_merged_dataset.csv')

# Display the first few rows and basic statistical summary
display(df.head())

# ---------------------------------------------------------
# Plotting the National Inflation Trend
# ---------------------------------------------------------
# Group by year and calculate the mean average income across ALL municipalities
# This creates our baseline "National Trend"
national_trend = df.groupby('Year')['Average Income'].mean().reset_index()

# Filter for the years 2008 to 2024 as mentioned in the text
national_trend = national_trend[(national_trend['Year'] >= 2008) & (national_trend['Year'] <= 2024)]

# Create the line chart
fig_trend = px.line(
    national_trend, 
    x='Year', 
    y='Average Income',
    title="National Average Income Trend (2008-2024)",
    labels={'Average Income': 'Average Income (DKK)', 'Year': 'Year'},
    markers=True
)

# Format the y-axis to include commas for easier reading of large numbers
fig_trend.update_layout(yaxis_tickformat=',')
fig_trend.show()

,Municipality,Year,Average Income,Employed_Ratio,Unemployed_Ratio,Enrolled_Ratio,Outside_Ratio
0,Aabenraa,2008,257029.5,0.587436,0.016870,0.110913,0.284782
1,Aabenraa,2009,255130.2,0.558074,0.034676,0.114738,0.292512
2,Aabenraa,2010,271481.9,0.547915,0.034071,0.121560,0.296454
3,Aabenraa,2011,268613.8,0.536807,0.035979,0.123741,0.303473
4,Aabenraa,2012,268157.0,0.529620,0.039931,0.124723,0.305727


## 3. Data Analysis

### Part 1: Geographical Distribution & Socioeconomic Anomalies ([Your Name])
Our analysis of the merged HFUDD16 and IFOR35 datasets yielded several key insights:
1. **Demographic Uniformity:** We hypothesized that wealthier municipalities would have drastically different employment/enrollment ratios. Surprisingly, the data showed that these ratios are relatively uniform across the country. Income disparities are therefore largely driven by the *type* of local industry, rather than the raw availability of workers.
2. **The Vejen & Billund Anomalies:** While tracking municipal income over time, we discovered a massive localized shock. Between 2022 and 2023, average incomes spiked dramatically in Vejen (from 314k to 446k DKK) and Billund (from 329k to 470k DKK). 
3. **Divergent Corrections:** Following the 2023 spike, Billund's average income corrected heavily, dropping back down to 340k DKK the following year. Vejen, however, maintained its new baseline, stabilizing at 435k DKK in 2024.

### Part 2: Industry vs. Income ([Groupmate 1 Name])
*(Groupmate 1: Describe your findings regarding how company locations impact the wealth of specific municipalities. Did you find clusters?)*

### Part 3: Education to Industry Pipeline ([Groupmate 2 Name])
*(Groupmate 2: Describe your findings on how specific education levels filter into different corporate sectors based on the Sankey data).*

In [3]:
# ---------------------------------------------------------
# Plotting the Vejen & Billund Anomaly
# ---------------------------------------------------------
# Filter the main dataframe for only Vejen and Billund, and focus on recent years
df_anomaly = df[
    (df['Municipality'].isin(['Vejen', 'Billund'])) & 
    (df['Year'] >= 2021)
].copy()

# Group by Year and Municipality to get the clean average 
# (This acts as a safeguard in case your merged dataset still has multiple rows per municipality per year)
df_anomaly_grouped = df_anomaly.groupby(['Year', 'Municipality'])['Average Income'].mean().reset_index()

# Create the line chart comparing the two municipalities
fig_anomaly = px.line(
    df_anomaly_grouped, 
    x='Year', 
    y='Average Income', 
    color='Municipality', 
    markers=True,
    title="The 2023 Income Spike: Vejen vs. Billund",
    labels={'Average Income': 'Average Income (DKK)', 'Year': 'Year'}
)

# Apply styling to match the previous chart
fig_anomaly.update_layout(yaxis_tickformat=',')
fig_anomaly.show()

# (Groupmates: Add the code for your specific data analysis steps below this)

## 4. Genre

We structured our final webpage using the **Partitioned Poster** genre, layered over a **Martini Glass** narrative structure.

### Visual Narrative (Tools from Segel & Heer)
* **Visual Structuring - Consistent Visual Platform:** We used Plotly across all visualizations to maintain a cohesive look, feel, and interaction logic. 
* **Highlighting - Feature Distinction & Close-ups:** In the interactive dashboard, clicking a municipality acts as a close-up, filtering the ternary plot to distinguish that specific region's features against the rest of the dataset.
* **Transition Guidance - Familiar Objects:** We used a geographic map of Denmark as our primary anchor. It is immediately recognizable to the user and serves as an intuitive navigation menu for complex data.

### Narrative Structure (Tools from Segel & Heer)
* **Ordering - Linear:** The webpage is structured linearly (The "Stem" of the Martini Glass). Users read through structured insights (National trends -> Industries -> Education flows) moving vertically down the page.
* **Interactivity - Filtering / Selection:** At the end of each linear section, we provide the "Bowl" of the Martini glass. The Plotly Dash app allows for complex cross-filtering, empowering the user to drive the final state of the visualization.
* **Messaging - Captions / Multi-messaging:** Each visualization is accompanied by a partitioned text block that explicitly explains the insights the user should draw from the tool.

## 5. Visualizations

### The Interactive Map & Ternary Plot ([Your Name])
* **The Map (Choropleth):** Ideal for showing the spatial distribution of wealth. It allows users to immediately spot regional clustering of high or low incomes.
* **The Ternary Plot:** We chose this because socioeconomic status (Employed, Unemployed, Enrolled) is *compositional data* (parts of a whole). A ternary plot effectively visualizes these three ratios simultaneously.
* **Why they are right for the story:** By cross-filtering these two plots, the user physically experiences our primary finding: clicking across vastly different income colors on the map results in very little movement on the ternary plot, proving demographic uniformity.

### Industry Distribution Plot ([Groupmate 1 Name])
*(Groupmate 1: Explain your Plotly visualization here. Why a scatter/bar/bubble chart?)*

### Education Flow Sankey Diagram ([Groupmate 2 Name])
*(Groupmate 2: Explain your Sankey diagram here. Why is Sankey the best way to show the flow from universities to specific industries?)*

## 6. Discussion

**What went well?**
We successfully integrated disparate datasets (spatial borders, income histories, and education levels) into a cohesive narrative. Overcoming the technical hurdle of hosting a live, interactive Python backend (Render) and embedding it into a static GitHub Pages site was a major success, allowing us to achieve true two-way cross-filtering.

**What is still missing / What could be improved?**
While we identified major anomalies (like the Vejen/Billund income spikes), our current dataset cannot conclusively tell us *why* it happened. Is it due to a single major company paying out massive bonuses one year? Did a wealthy demographic suddenly move? To improve this project, we would need highly granular, historical tracking of specific company payouts and individual taxpayer migrations. Furthermore, our company dataset only accounts for the "main" companies, potentially skewing our view of the SME (Small and Medium Enterprise) impact on local economies.

## 7. Contributions

While all group members participated in conceptualizing the data story and formatting the final GitHub webpage, the primary technical responsibilities were divided as follows:

* **Andrea De Pascale (s243094):** Led Part 1. Responsible for the loading via API calls and merging of the HFUDD16 and IFOR35 datasets, normalizing municipality GeoJSON data, and building the interactive Dash application. Managed the cloud deployment on Render to enable two-way cross-filtering between the Choropleth map and Ternary plot.
* **[Groupmate 1 Name]:** Led Part 2. Responsible for cleaning and integrating the company location dataset, conducting the exploratory analysis on corporate impact, and generating the static interactive visualizations relating industry to municipal wealth.
* **[Groupmate 2 Name]:** Led Part 3. Responsible for mapping the pathways from education levels to industry sectors. Designed and coded the complex Sankey diagrams to visualize the flow of the workforce.

## 8. References

1. **Statistics Denmark (Danmarks Statistik):**
   * *HFUDD16 Dataset*: Population by education, socioeconomic status, and municipality.
   * *IFOR35 Dataset*: Average income by municipality and deciles.
2. **Segel, E., & Heer, J. (2010):** *Narrative Visualization: Telling Stories with Data*. IEEE Transactions on Visualization and Computer Graphics. (Used for genre and structural framework).
3. **Click That Hood (GitHub):** Open-source repository used for the GeoJSON borders of Danish municipalities.